# OnePulse Chart Generation

This notebook provides an interactive way to inspect the data at each stage of the chart generation process.

In [1]:
# 1. Imports
import sys
import os

# Add the parent directory to the Python path
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

from src import config
from src.data_loader import load_summary_data, load_raw_data
from src.data_processor import process_raw_audience_data #, get_combined_data
from src.ppt_generator import generate_presentation

# Display up to 100 columns
import pandas as pd
pd.set_option('display.max_columns', 100)

In [2]:
# 2. Load summary data

#  Note - this is no longer being used for chart data
# - only being loaded so it can be used for validation
# of the processed respondent level data
summary_data, summary_counts = load_summary_data()
print(f"Loaded {len(summary_data)} summary charts")

# Display first summary chart data
if summary_data:
    first_title = next(iter(summary_data))
    print(f"\nExample - {first_title}:")
    cats, vals = summary_data[first_title]
    for cat, val in zip(cats, vals):
        print(f"{cat}: {val:.1%}")

Loaded 2 summary charts

Example - Thinking about Lloyds Bank; if you were considering investing; which of these phrases best describes how you feel about using them as a provider of investment products?  (Exclusive):
I’d consider them equally with others: 29.7%
It’s one of the first I’d consider using: 29.1%
It’s the first I’d consider using: 12.8%
I might consider them but I’d consider others first: 12.4%
I probably wouldn’t consider them but I wouldn’t rule them out: 9.2%
Don't know: 3.6%
I definitely wouldn’t consider them: 3.2%


In [3]:
for i, data in enumerate(summary_data.keys()):
    print(f"Question {i+1}:")
    print(data)

Question 1:
Thinking about Lloyds Bank; if you were considering investing; which of these phrases best describes how you feel about using them as a provider of investment products?  (Exclusive)
Question 2:
We would now like your opinion of Lloyds Bank as an investment product provider. Do not worry if you are not very familiar with them; it is your impressions we are interested in.Which of the following statements do you agree with. Select all that apply. (Multiple)


In [4]:
# 3. Load raw data
# (Note - Mapping is now redundant, but kept for reference)
raw_df, mapping = load_raw_data()
print(f"Raw data shape: {raw_df.shape if raw_df is not None else 'None'}")
print(f"Mapping shape: {mapping.shape if mapping is not None else 'None'}")

# Display mapping
display(mapping)

# Display raw_df
# display(raw_df)

Raw data shape: (501, 32)
Mapping shape: (11, 2)


,key,type
0,Total,summary
1,"Bank(s): HSBC, Santander, Barclays, RBS, Natwe...",summary
2,Gender: Male,raw
3,Gender: Female,raw
4,"Home location: South East, Greater London [Lon...",raw
5,Home location: Greater London / Gender: Female...,raw
6,Bank(s): Lloyds [Lloyds customers],raw
7,Chart combining row 2 and row 7,raw
12,segment_name,column_name
13,Lloyds customers,Bank(s)


In [5]:
# First, let's see what banks are in the data
bank_column = 'Bank(s)'
if bank_column in raw_df.columns:
    # Get all unique banks from the semicolon-separated lists
    all_banks = set()
    for banks_str in raw_df[bank_column].dropna():
        banks = [bank.strip() for bank in banks_str.split(';')]
        all_banks.update(banks)
    
    print("Found these banks in the data:")
    for bank in sorted(all_banks):
        print(f"- {bank}")
    
    # Create boolean columns for each bank
    for bank in all_banks:
        column_name = f"{bank.lower().replace(' ', '_')}_customer"
        raw_df[column_name] = raw_df[bank_column].fillna('').str.contains(bank, regex=False)
    
    print("\nCreated these new columns:")
    for bank in sorted(all_banks):
        print(f"- {bank.lower().replace(' ', '_')}_customer")

Found these banks in the data:
- Atom
- Barclays
- HSBC
- Halifax
- Lloyds
- Metro
- Monzo
- Nationwide
- Natwest
- Other
- RBS
- Revolut
- Santander
- Starling
- TSB

Created these new columns:
- atom_customer
- barclays_customer
- hsbc_customer
- halifax_customer
- lloyds_customer
- metro_customer
- monzo_customer
- nationwide_customer
- natwest_customer
- other_customer
- rbs_customer
- revolut_customer
- santander_customer
- starling_customer
- tsb_customer


In [6]:
# Now we've got a column for each bank's customers.
raw_df.head()

3,User ID,Created,Home location,Bank(s),Age,Gender,Age range,Device type,Region,Respondent ID,Q(1_1) Lloyds Bank[Question: Which of the following are you aware offer investment products (e.g. stocks and shares ISAs and ‘ready made investments’)? Please tick all that apply.],Q(1_2) Halifax[Question: Which of the following are you aware offer investment products (e.g. stocks and shares ISAs and ‘ready made investments’)? Please tick all that apply.],Q(1_3) Barclays[Question: Which of the following are you aware offer investment products (e.g. stocks and shares ISAs and ‘ready made investments’)? Please tick all that apply.],Q(1_4) HSBC[Question: Which of the following are you aware offer investment products (e.g. stocks and shares ISAs and ‘ready made investments’)? Please tick all that apply.],Q(1_5) Santander[Question: Which of the following are you aware offer investment products (e.g. stocks and shares ISAs and ‘ready made investments’)? Please tick all that apply.],Q(1_6) Nationwide[Question: Which of the following are you aware offer investment products (e.g. stocks and shares ISAs and ‘ready made investments’)? Please tick all that apply.],Q(1_7) Hargreaves Lansdown[Question: Which of the following are you aware offer investment products (e.g. stocks and shares ISAs and ‘ready made investments’)? Please tick all that apply.],Q(1_8) Moneybox [Question: Which of the following are you aware offer investment products (e.g. stocks and shares ISAs and ‘ready made investments’)? Please tick all that apply.],Q(1_9) A J Bell [Question: Which of the following are you aware offer investment products (e.g. stocks and shares ISAs and ‘ready made investments’)? Please tick all that apply.],Q(1_10) Interactive Investor[Question: Which of the following are you aware offer investment products (e.g. stocks and shares ISAs and ‘ready made investments’)? Please tick all that apply.],Q(1_11) Nutmeg[Question: Which of the following are you aware offer investment products (e.g. stocks and shares ISAs and ‘ready made investments’)? Please tick all that apply.],Q(1_12) None of these[Question: Which of the following are you aware offer investment products (e.g. stocks and shares ISAs and ‘ready made investments’)? Please tick all that apply.],Q(1) Comments [Question: Which of the following are you aware offer investment products (e.g. stocks and shares ISAs and ‘ready made investments’)? Please tick all that apply.],Q(2) Thinking about Lloyds Bank; if you were considering investing; which of these phrases best describes how you feel about using them as a provider of investment products?,Q(2) Comments [Question: Thinking about Lloyds Bank; if you were considering investing; which of these phrases best describes how you feel about using them as a provider of investment products? ],Q(3_1) I would trust Lloyds Bank to look after my investments[Question: We would now like your opinion of Lloyds Bank as an investment product provider. Do not worry if you are not very familiar with them; it is your impressions we are interested in.Which of the following statements do you agree with. Select all that apply.],Q(3_2) It would be easy to start investing with Lloyds Bank [Question: We would now like your opinion of Lloyds Bank as an investment product provider. Do not worry if you are not very familiar with them; it is your impressions we are interested in.Which of the following statements do you agree with. Select all that apply.],Q(3_3) Lloyds Bank would give me access to a wide range of investment options[Question: We would now like your opinion of Lloyds Bank as an investment product provider. Do not worry if you are not very familiar with them; it is your impressions we are interested in.Which of the following statements do you agree with. Select all that apply.],Q(3_4) Investing with Lloyds Bank would be good value[Question: We would now like your opinion of Lloyds Bank as an investment product provider. Do not worry if you are not very familiar with the

In [7]:
# Load audience definitions from JSON file
import json

def apply_audience_filter(df, definition):
    if isinstance(definition, dict):
        if "AND" in definition:
            masks = [apply_audience_filter(df, cond) for cond in definition["AND"]]
            return pd.concat(masks, axis=1).all(axis=1)
        elif "OR" in definition:
            masks = [apply_audience_filter(df, cond) for cond in definition["OR"]]
            return pd.concat(masks, axis=1).any(axis=1)
        else:
            # Simple case: {column: value}
            # (assumes only one key at this level)
            for col, val in definition.items():
                return df[col] == val
    else:
        raise ValueError("Invalid audience definition")

with open("audience_segments.json") as f:
    audience_defs = json.load(f)

audience_dfs = {}
for name, definition in audience_defs.items():
    mask = apply_audience_filter(raw_df, definition)
    audience_dfs[name] = raw_df[mask]
    print(f"{name}: {audience_dfs[name].shape[0]} respondents")

Lloyds Customers: 206 respondents
Non-Lloyds Customers: 295 respondents
Male 18-34: 151 respondents


In [8]:
# Process all questions in the raw data
results = process_raw_audience_data(raw_df)

Found 3 questions to process

Processing Q1 as multi_select
Found 12 data columns and 1 comment columns
Found categories: ['Lloyds Bank', 'Halifax', 'Barclays', 'HSBC', 'Santander', 'Nationwide', 'Hargreaves Lansdown', 'Moneybox', 'A J Bell', 'Interactive Investor', 'Nutmeg', 'None of these']
Successfully processed Q1
Values: [np.float64(0.3812375249500998), np.float64(0.3812375249500998), np.float64(0.36926147704590817), np.float64(0.3473053892215569), np.float64(0.34331337325349304), np.float64(0.3393213572854291), np.float64(0.2654690618762475), np.float64(0.26147704590818366), np.float64(0.2055888223552894), np.float64(0.19760479041916168), np.float64(0.15169660678642716), np.float64(0.06786427145708583)]

Processing Q2 as single_select
Found 1 data columns and 1 comment columns
Found categories: ['I’d consider them equally with others', 'I probably wouldn’t consider them but I wouldn’t rule them out', 'It’s one of the first I’d consider using', 'It’s the first I’d consider using',

In [9]:
results

[('1',
  ['Barclays',
   'Santander',
   'Lloyds Bank',
   'Halifax',
   'HSBC',
   'Nationwide',
   'Hargreaves Lansdown',
   'Moneybox',
   'Nutmeg',
   'A J Bell',
   'Interactive Investor',
   'None of these'],
  [np.float64(0.3812375249500998),
   np.float64(0.3812375249500998),
   np.float64(0.36926147704590817),
   np.float64(0.3473053892215569),
   np.float64(0.34331337325349304),
   np.float64(0.3393213572854291),
   np.float64(0.2654690618762475),
   np.float64(0.26147704590818366),
   np.float64(0.2055888223552894),
   np.float64(0.19760479041916168),
   np.float64(0.15169660678642716),
   np.float64(0.06786427145708583)]),
 ('2',
  ['I’d consider them equally with others',
   'It’s one of the first I’d consider using',
   'It’s the first I’d consider using',
   'I might consider them but I’d consider others first',
   'I probably wouldn’t consider them but I wouldn’t rule them out',
   "Don't know",
   'I definitely wouldn’t consider them'],
  [np.float64(0.2974051896207584

In [10]:
# raw_audience_data = []
# for q_id, categories, values in results:
#     # Extract the full question text as before
#     q_cols = [col for col in raw_df.columns if col.startswith(f"Q({q_id}_")]
#     if q_cols:
#         question_text = q_cols[0].split('[', 1)[1].rstrip(']')
#         title = question_text
#     else:
#         title = f"Q{q_id}"
#     raw_audience_data.append(
#         (title, categories, [("Total", values, len(raw_df))])
#     )

In [11]:
# 5 Process combined data
# combined_data = get_combined_data(raw_df, summary_data, summary_counts, mapping)


# print(f"Processed {len(combined_data)} combined charts")

# # Display first combined chart data
# if combined_data:
#     first_title, categories, segments = combined_data[0]
#     print(f"\nExample - {first_title}:")
#     for label, vals, n_resp in segments:
#         print(f"\n{label} (n={n_resp}):")
#         for cat, val in zip(categories, vals):
#             print(f"{cat}: {val:.1%}")

In [12]:
from src.data_processor import get_combined_data_from_audiences

combined_data = get_combined_data_from_audiences(raw_df, results, audience_dfs)
print(f"Processed {len(combined_data)} combined charts")

# Display first combined chart data
if combined_data:
    first_title, categories, segments = combined_data[0]
    print(f"\nExample - {first_title}:")
    for label, vals, n_resp in segments:
        print(f"\n{label} (n={n_resp}):")
        for cat, val in zip(categories, vals):
            print(f"{cat}: {val:.1%}")

Processed 3 combined charts

Example - Lloyds Bank[Question: Which of the following are you aware offer investment products (e.g. stocks and shares ISAs and ‘ready made investments’)? Please tick all that apply.]:

Total (n=501):
Barclays: 36.9%
Santander: 34.7%
Lloyds Bank: 38.1%
Halifax: 34.3%
HSBC: 38.1%
Nationwide: 33.9%
Hargreaves Lansdown: 26.5%
Moneybox: 26.1%
Nutmeg: 19.8%
A J Bell: 15.2%
Interactive Investor: 20.6%
None of these: 6.8%

Lloyds Customers (n=206):
Barclays: 39.3%
Santander: 34.0%
Lloyds Bank: 33.0%
Halifax: 32.0%
HSBC: 36.4%
Nationwide: 32.5%
Hargreaves Lansdown: 22.3%
Moneybox: 23.3%
Nutmeg: 18.9%
A J Bell: 16.5%
Interactive Investor: 18.9%
None of these: 1.5%

Non-Lloyds Customers (n=295):
Barclays: 35.3%
Santander: 35.3%
Lloyds Bank: 41.7%
Halifax: 35.9%
HSBC: 39.3%
Nationwide: 34.9%
Hargreaves Lansdown: 29.5%
Moneybox: 28.1%
Nutmeg: 20.3%
A J Bell: 14.2%
Interactive Investor: 21.7%
None of these: 10.5%

Male 18-34 (n=151):
Barclays: 28.5%
Santander: 28.5%
Llo

In [13]:
combined_data

[('Lloyds Bank[Question: Which of the following are you aware offer investment products (e.g. stocks and shares ISAs and ‘ready made investments’)? Please tick all that apply.]',
  ['Barclays',
   'Santander',
   'Lloyds Bank',
   'Halifax',
   'HSBC',
   'Nationwide',
   'Hargreaves Lansdown',
   'Moneybox',
   'Nutmeg',
   'A J Bell',
   'Interactive Investor',
   'None of these'],
  [('Total',
    [np.float64(0.36926147704590817),
     np.float64(0.3473053892215569),
     np.float64(0.3812375249500998),
     np.float64(0.34331337325349304),
     np.float64(0.3812375249500998),
     np.float64(0.3393213572854291),
     np.float64(0.2654690618762475),
     np.float64(0.26147704590818366),
     np.float64(0.19760479041916168),
     np.float64(0.15169660678642716),
     np.float64(0.2055888223552894),
     np.float64(0.06786427145708583)],
    501),
   ('Lloyds Customers',
    [np.float64(0.3932038834951456),
     np.float64(0.33980582524271846),
     np.float64(0.3300970873786408),
   

In [14]:
# SKIPPING FOR NOW...

# from src.data_processor import extract_categories_from_columns, process_single_select_question

# # Example: for a specific question (e.g., Q2)
# q_id = "2"
# categories = extract_categories_from_columns(non_lloyds_df, q_id, is_multi_select=False)
# values = process_single_select_question(non_lloyds_df, q_id, categories)
# n_resp = len(non_lloyds_df)

# # Append to the combined data in the new format
# combined_data.append((
#     "*True* Non-Lloyds customers: " + "HARD CODED CHART TITLE",
#     categories,
#     [("Total", values, n_resp)]
# ))

In [15]:
# # Create a DataFrame for non-Lloyds customers
# non_lloyds_df = raw_df[raw_df['lloyds_customer'] == False]

# # Append to the combined data
# combined_data.append((
#     "*True* Non-Lloyds customers: " + "HARD CODED CHART TITLE",
#     [("Total", values, n_resp)]
# ))

# title = "Non-Lloyds customers: " + "HARD CODED CHART TITLE"

# combined_data.append((title, categories, segments))

In [16]:
for entry in combined_data[:3]:
    print(entry)

('Lloyds Bank[Question: Which of the following are you aware offer investment products (e.g. stocks and shares ISAs and ‘ready made investments’)? Please tick all that apply.]', ['Barclays', 'Santander', 'Lloyds Bank', 'Halifax', 'HSBC', 'Nationwide', 'Hargreaves Lansdown', 'Moneybox', 'Nutmeg', 'A J Bell', 'Interactive Investor', 'None of these'], [('Total', [np.float64(0.36926147704590817), np.float64(0.3473053892215569), np.float64(0.3812375249500998), np.float64(0.34331337325349304), np.float64(0.3812375249500998), np.float64(0.3393213572854291), np.float64(0.2654690618762475), np.float64(0.26147704590818366), np.float64(0.19760479041916168), np.float64(0.15169660678642716), np.float64(0.2055888223552894), np.float64(0.06786427145708583)], 501), ('Lloyds Customers', [np.float64(0.3932038834951456), np.float64(0.33980582524271846), np.float64(0.3300970873786408), np.float64(0.32038834951456313), np.float64(0.3640776699029126), np.float64(0.32524271844660196), np.float64(0.2233009708

In [17]:
# Convert results to the expected format for generate_presentation
raw_audience_data = []
for q_id, categories, values in results:
    # Try to extract the full question text for this q_id
    q_cols = [col for col in raw_df.columns if f"Q({q_id}" in col and 'Comments' not in col]
    if q_cols:
        # Extract the question text from the column name
        # Format is: "Q(2) Question text here"
        try:
            # Split on the first closing parenthesis and take everything after it
            question_text = q_cols[0].split(')', 1)[1].strip()
            title = question_text
        except IndexError:
            # If no closing parenthesis found, use the full column name
            title = q_cols[0]
    else:
        # Fallback: use Q{q_id}
        title = f"Q{q_id}"
    raw_audience_data.append(
        (title, categories, [("Total", values, len(raw_df))])
    )

In [18]:
raw_audience_data

[('Lloyds Bank[Question: Which of the following are you aware offer investment products (e.g. stocks and shares ISAs and ‘ready made investments’)? Please tick all that apply.]',
  ['Barclays',
   'Santander',
   'Lloyds Bank',
   'Halifax',
   'HSBC',
   'Nationwide',
   'Hargreaves Lansdown',
   'Moneybox',
   'Nutmeg',
   'A J Bell',
   'Interactive Investor',
   'None of these'],
  [('Total',
    [np.float64(0.3812375249500998),
     np.float64(0.3812375249500998),
     np.float64(0.36926147704590817),
     np.float64(0.3473053892215569),
     np.float64(0.34331337325349304),
     np.float64(0.3393213572854291),
     np.float64(0.2654690618762475),
     np.float64(0.26147704590818366),
     np.float64(0.2055888223552894),
     np.float64(0.19760479041916168),
     np.float64(0.15169660678642716),
     np.float64(0.06786427145708583)],
    501)]),
 ('Thinking about Lloyds Bank; if you were considering investing; which of these phrases best describes how you feel about using them as 

In [19]:
# Check of what we're charting, and where the data is coming from;

# print(f"Adding {len(summary_data)} summary slides")
print(f"Adding {len(raw_audience_data)} raw audience slides")
print(f"Adding {len(combined_data)} combined slides")


# for data in summary_data:
#     print(f"### Summary Data ###")
#     print(data)

for data in raw_audience_data:
    print(f"### Raw Audience Data ###")
    print(data)

for data in combined_data:
    print(f"### Combined Data ###")
    print(data)


Adding 3 raw audience slides
Adding 3 combined slides
### Raw Audience Data ###
('Lloyds Bank[Question: Which of the following are you aware offer investment products (e.g. stocks and shares ISAs and ‘ready made investments’)? Please tick all that apply.]', ['Barclays', 'Santander', 'Lloyds Bank', 'Halifax', 'HSBC', 'Nationwide', 'Hargreaves Lansdown', 'Moneybox', 'Nutmeg', 'A J Bell', 'Interactive Investor', 'None of these'], [('Total', [np.float64(0.3812375249500998), np.float64(0.3812375249500998), np.float64(0.36926147704590817), np.float64(0.3473053892215569), np.float64(0.34331337325349304), np.float64(0.3393213572854291), np.float64(0.2654690618762475), np.float64(0.26147704590818366), np.float64(0.2055888223552894), np.float64(0.19760479041916168), np.float64(0.15169660678642716), np.float64(0.06786427145708583)], 501)])
### Raw Audience Data ###
('Thinking about Lloyds Bank; if you were considering investing; which of these phrases best describes how you feel about using them 

In [20]:
# Now call the PPT generator
custom_output = os.path.join(config.BASE_DIR, 'OnePulse_Summary_Notebook_version_no_summary_data2.pptx')

generate_presentation(
    raw_audience_data, combined_data, output_path=custom_output
)

Saved → /Users/sthompso/Library/CloudStorage/OneDrive-PublicisGroupe/Projects/OnePulse Charting (Jen)/2025-05-21/OnePulse_Automation/OnePulse_Summary_Notebook_version_no_summary_data2.pptx
